# 🎙️ Voicebox: Qwen3-TTS Engine.
This uses [voicebox.sh](https://voicebox.sh) and the Qwen3 TTS engine. to generate Text to voice, then (optionally) save the output to your Google Drive.

This does REQUIRES you to have a voice profile, either from using the tool at the bottom of this notebook or [from an existing install of voicebox.sh](https://docs.voicebox.sh/overview/quick-start)

However that feature of Voicebox requires almost no compute power, and can be done on any hardware.

If you have a HUGE amount of text to convert to speech, This sytem may crash, It is HIGHLY recomended to use the save to Google Drive feature, as it will save each chunk it has processed. So you can just listen to the last section that was generated, and resume the process from there.

In [ ]:
# @title 🎙️ Voicebox: Qwen3-TTS Direct Generator

# @markdown ### ⚙️️ **Model & Drive Settings** <br />
# @markdown Higher quality outputs take about 3X longer to load. Quality difference is marginal.
higher_quality_outputs = False # @param {type:"boolean"}
# @markdown Check this box to save to your Google Drive. Uncheck to save temporarily in the Colab runtime and auto-download.
save_to_drive = False # @param {type:"boolean"}
# @markdown Output will be saved under /Voicebox_Outputs/Project_name/
project_name = "Project_name" # @param ["Project_name"] {allow-input: true}

# @markdown ---
# @markdown ### 📝 **Text Input**
text_to_speak = "Oh! Hello there! Wait, can you actually hear me?! Yay, I have a voice now!" # @param ["Oh! Hello there! Wait, can you actually hear me?! Yay, I have a voice now!"] {allow-input: true}

# @markdown ---
# @markdown ### 📁 **Voice Profile** <br />
# @markdown You need a voice profile for this to work, as luck would have it you can find a tool to make one down below! <br />
# @markdown By default this will re-use the same voice profile for this whole session. Untick this to be prompted for upload every time.
reuse_voice_profile = True # @param {type:"boolean"}
# @markdown Your voice profile must contain a transcript of the example, if it does not, you can manually add that here. <br />
manual_ref_text = "" # @param {type:"string"}

# @markdown ---
# @markdown ### Advanced Settings
# @markdown How many characters to process at once. Lower is more stable processing, Higher is better continuity in the tone of voice. <br />
# @markdown Note: If your total number of characters to be spoken is less than this setting, it just processes everything at once.
chunk_size = 1000 # @param {type:"slider", min:100, max:5000, step:50}

import sys
import os
import shutil
import zipfile
import json
import time
import re
import glob

# 1. System Dependencies & Repository Setup
print("⏳ Step 1/5: Installing system packages and cloning Voicebox repository...")
!apt-get install -y -qq sox libsox-fmt-all > /dev/null 2>&1
!rm -rf /content/voicebox
!git clone https://github.com/jamiepine/voicebox.git /content/voicebox

# 2. Python Package Installation
print("📦 Step 2/5: Installing core Python dependencies...")
!pip install -q -U qwen-tts accelerate soundfile

# 3. Imports & Logging Configuration
import torch
import numpy as np
import soundfile as sf
import transformers
import warnings

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

from IPython.display import Audio, display, clear_output
from google.colab import files
from qwen_tts import Qwen3TTSModel

print("✅ Setup complete! Preparing Output Folder & Voice Profile...")

# 4. Configure Output Path & Mount Drive (Optional)
clear_output()
if save_to_drive:
    print("\n📂 Step 3/5: Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    base_output_dir = f'/content/drive/MyDrive/Voicebox_Outputs/{project_name}/'
else:
    print("\n📂 Step 3/5: Using local Colab runtime for saving...")
    base_output_dir = f'/content/Voicebox_Outputs/{project_name}/'

os.makedirs(base_output_dir, exist_ok=True)
print(f"✅ Output folder ready: {base_output_dir}")

# 5. Interactive Upload Logic with Re-use capability
clear_output(wait=True)
if reuse_voice_profile and 'global_profile_zip_path' in globals() and os.path.exists(global_profile_zip_path):
    print(f"♻️ Step 4/5: Reusing previously uploaded voice profile: {global_profile_zip_path}")
    profile_zip_path = global_profile_zip_path
else:
    print("⬆️ Step 4/5: Please upload your Voicebox profile (.zip):")
    uploaded = files.upload()
    if uploaded:
        uploaded_filename = list(uploaded.keys())[0]
        profile_zip_path = os.path.join('/content', uploaded_filename)
        global_profile_zip_path = profile_zip_path # Store globally for next run
        print(f"✅ Uploaded successfully: {profile_zip_path}")
    else:
        raise FileNotFoundError("❌ No .zip file provided.")

# 6. Load Qwen3-TTS Model
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base" if higher_quality_outputs else "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
model_label = "1.7B" if higher_quality_outputs else "0.6B"

print(f"\n🧠 Step 5/5: Loading Qwen3-TTS {model_label} Model into GPU VRAM...")

# Choose bfloat16 if supported, otherwise float32 to prevent CUDA assertion faults
target_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

model = Qwen3TTSModel.from_pretrained(
    model_id,
    torch_dtype=target_dtype,
    device_map="auto"
)

print(f"✅ {model_label} Model loaded into GPU VRAM successfully!")

# 7. Text Chunking Helper
def chunk_text(text, max_chars=300):
    sentences = re.split(r'(?<=[.!?\n])\s+', text)
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chars:
            current_chunk += sentence + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

# 8. Unpack Voice Profile & Run TTS Generation
print("\n🎙️ Unpacking Voice Profile & Generating Speech...")
extract_path = "/content/temp_voice_profile"
os.makedirs(extract_path, exist_ok=True)

try:
    with zipfile.ZipFile(profile_zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    ref_audio = None
    ref_text = manual_ref_text.strip() if manual_ref_text else ""

    # Locate .wav audio
    for root, dirs, f_list in os.walk(extract_path):
        for file in f_list:
            if file.endswith(".wav") and not ref_audio:
                ref_audio = os.path.join(root, file)

    # Locate transcript in samples.json
    if not ref_text and ref_audio:
        audio_basename = os.path.basename(ref_audio)
        for root, dirs, f_list in os.walk(extract_path):
            if "samples.json" in f_list:
                try:
                    with open(os.path.join(root, "samples.json"), 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        if isinstance(data, list):
                            for item in data:
                                if isinstance(item, dict) and "text" in item:
                                    if "audio" in item and audio_basename in item["audio"]:
                                        ref_text = item["text"]
                                        break
                                    if not ref_text:
                                        ref_text = item["text"]
                except Exception as e:
                    print(f"⚠️ Could not parse samples.json: {e}")

    if not ref_audio:
        raise FileNotFoundError("❌ No reference .wav audio found inside the .zip profile.")

    # Configure outputs
    timestamp = int(time.time())
    output_filename = os.path.join(base_output_dir, f"voicebox_output_{timestamp}.wav")

    # Create sub-directory for chunks
    chunks_dir = os.path.join(base_output_dir, f"chunks_{timestamp}")
    os.makedirs(chunks_dir, exist_ok=True)

    text_chunks = chunk_text(text_to_speak, max_chars=chunk_size)
    all_audio_arrays = []
    final_sample_rate = 24000

    print(f"⚡ Generating speech across {len(text_chunks)} chunk(s)...")
    print(f"📁 Chunks will be saved to: {chunks_dir}")

    for i, chunk in enumerate(text_chunks, 1):
        if not chunk.strip():
            continue
        print(f"   └─ Processing chunk {i}/{len(text_chunks)}...")
        if ref_text:
            wavs, sr = model.generate_voice_clone(
                text=chunk,
                language="English",
                ref_audio=ref_audio,
                ref_text=ref_text
            )
        else:
            wavs, sr = model.generate_voice_clone(
                text=chunk,
                language="English",
                ref_audio=ref_audio,
                x_vector_only_mode=True
            )

        chunk_audio = wavs[0]
        all_audio_arrays.append(chunk_audio)
        final_sample_rate = sr

        # Save individual chunk
        chunk_filename = os.path.join(chunks_dir, f"chunk_{i:03d}.wav")
        sf.write(chunk_filename, chunk_audio, sr)

    if all_audio_arrays:
        final_audio = np.concatenate(all_audio_arrays)
        sf.write(output_filename, final_audio, final_sample_rate)
        print(f"\n🎉 Success! Combined audio saved to: {output_filename}\n")
        display(Audio(output_filename))

        # 9. Zip and Download logic if not saving to drive
        if not save_to_drive:
            print("\n📦 Zipping outputs for download...")
            zip_filename = f"/content/Voicebox_Output_{timestamp}.zip"

            with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
                # Add the combined main audio file
                zipf.write(output_filename, arcname=os.path.basename(output_filename))
                # Add the chunks directory and its contents
                for root, dirs, files_in_dir in os.walk(chunks_dir):
                    for file in files_in_dir:
                        file_path = os.path.join(root, file)
                        arcname = os.path.join(os.path.basename(chunks_dir), file)
                        zipf.write(file_path, arcname=arcname)

            print(f"⬇️ Downloading {os.path.basename(zip_filename)}...")
            files.download(zip_filename)

    else:
        print("❌ Generation failed. Check your input text.")

finally:
    if os.path.exists(extract_path):
        shutil.rmtree(extract_path)

In [ ]:
# @title 🔗 Voicebox: Merge Project Audio Files.

# @markdown ### 📁 **Select Project**
# @markdown Enter the project name to merge multiple audio files. Use the same Project_name as above! <br />
# @markdown This merges the files in alphabetical order of name, so will work fine aslong as the above is generated in the desired order.
project_name = "" # @param {type:"string"}
merged_output_filename = "merged_voicebox_output.wav" # @param {type:"string"}

import os
import glob
import re
import numpy as np
import soundfile as sf
from google.colab import drive

# 1. Mount Google Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Mounting Google Drive...")
    drive.mount('/content/drive')

# 2. Resolve Project Directory
target_project = project_name.strip() if project_name.strip() else "Project_name"
project_dir = f'/content/drive/MyDrive/Voicebox/{target_project}/'

print(f"🔍 Searching for audio files in: {project_dir}")

if not os.path.exists(project_dir):
    raise FileNotFoundError(f"❌ Project directory does not exist: {project_dir}")

# 3. Locate and Sort Output .wav Files
wav_files = [
    f for f in glob.glob(os.path.join(project_dir, "*.wav"))
    if not os.path.basename(f).startswith("merged_")
]

if not wav_files:
    raise FileNotFoundError(f"❌ No output .wav files found in {project_dir} to merge.")

# Sort alphabetically using natural sorting (so "part2" comes before "part10")
wav_files.sort(key=lambda x: [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', os.path.basename(x))])

print(f"Found {len(wav_files)} file(s) to merge in alphabetical order:")
for i, f in enumerate(wav_files, 1):
    print(f"   {i}. {os.path.basename(f)}")

# 4. Read and Concatenate Audio
audio_segments = []
target_sr = None

for file_path in wav_files:
    data, sr = sf.read(file_path)
    if target_sr is None:
        target_sr = sr
    elif sr != target_sr:
        print(f"⚠️ Warning: Sample rate mismatch on {os.path.basename(file_path)}. Resampling...")
        import librosa
        data = librosa.resample(data, orig_sr=sr, target_sr=target_sr)

    audio_segments.append(data)

# Combine into a single NumPy array
merged_audio = np.concatenate(audio_segments)

# 5. Save and Display Merged Output
save_path = os.path.join(project_dir, merged_output_filename)
sf.write(save_path, merged_audio, target_sr)

print(f"\n🎉 Successfully merged {len(wav_files)} files!")
print(f"💾 Saved to: {save_path}\n")

from IPython.display import Audio, display
display(Audio(save_path))

In [ ]:
# @title 🔗 Voicebox: Voice profile maker.


import os
import json
import uuid
import zipfile
from google.colab import files

# @markdown You can use this cell to make a voice profile for Cell 1. <br />
# @markdown You need four things. <br />
# @markdown 1. A name for the Voice, this can be anything. <br />
# @markdown 2. A version number (Optional, but helps you keep track!) <br />
# @markdown 3. A high quality sample of the voice with no background noise. This should be around 20-30 seconds long. (It will prompt you to upload your voice sample when you run this cell).<br />
# @markdown 4. An EXACT WORD-FOR-WORD transcript of what is said in your example audio.


# --- Interactive Colab Form Inputs ---
speaker_name = "My_Voice" # @param {type:"string"}
version = "v1" # @param {type:"string"}
transcript = "Enter the exact spoken text here." # @param {type:"string"}

print("Please upload your reference audio file (.wav):")
uploaded = files.upload()

if not uploaded:
    print("No file uploaded. Please run the cell again to upload a file.")
else:
    # 1. Prepare file paths and rename the audio
    original_audio_filename = list(uploaded.keys())[0]
    sample_uuid = str(uuid.uuid4())
    samples_dir = "samples"
    os.makedirs(samples_dir, exist_ok=True)
    new_audio_path = os.path.join(samples_dir, f"{sample_uuid}.wav")

    os.rename(original_audio_filename, new_audio_path)

    # 2. Generate the JSON metadata structures
    manifest_data = {
        "speaker": speaker_name,
        "version": version
    }

    samples_data = {
        f"samples/{sample_uuid}.wav": {
            "text": transcript
        }
    }

    # Write JSON files to disk
    with open("manifest.json", "w") as f:
        json.dump(manifest_data, f, indent=2)

    # JSON is natively compatible with JSON5, so we can duplicate it safely
    with open("manifest.json5", "w") as f:
        json.dump(manifest_data, f, indent=2)

    with open("samples.json", "w") as f:
        json.dump(samples_data, f, indent=2)

    # 3. Package everything into the ZIP archive
    formatted_name = speaker_name.lower().replace(" ", "-")
    zip_filename = f"profile-{formatted_name}-{version}.voicebox.zip"

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write("manifest.json")
        zipf.write("manifest.json5")
        zipf.write("samples.json")
        zipf.write(new_audio_path)

    print(f"\nSuccessfully packaged! Downloading {zip_filename}...")
    files.download(zip_filename)

    # 4. Cleanup temporary files from the Colab environment
    os.remove("manifest.json")
    os.remove("manifest.json5")
    os.remove("samples.json")
    os.remove(new_audio_path)
    os.rmdir(samples_dir)